In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [ ]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session

print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



In [3]:
# Read raw business data
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [4]:
# Extract the 'categories' column from business data
df_categories =df_read_business_data.select(F.col("categories"))

In [5]:
# Load data from dim_category
df_read_dim_category=spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")

In [7]:
# This function is used to get subcategory names from the 'categories' column

@F.udf(returnType=ArrayType(StringType())) # Define DataFrame schema
def get_category_name(text):

    if text is None:
       return ["Not Specified"]
    
    list_of_element = text.split(",")
     
    del list_of_element[0]

    if len(list_of_element) == 0:
        return  ["Not Specified"]
    
    list_of_element_without_space = [st.lower().strip() for st in list_of_element]

    return list_of_element_without_space



# Step 1: apply UDF, extract fields, then CACHE + materialize 

df_udf = (
    df_categories
    .withColumn("subcategory_name_list", get_category_name(F.col("categories")))
    .drop("new_eilte", "categories")
    .cache()
)


df_udf.count()  # Materialization happens at this stage

# Step 2: explode AFTER the UDF stage is fully materialized

df_subcategory_name = (
    df_udf
    .withColumn("subcategory_name", F.explode(F.col("subcategory_name_list")))
    .drop("subcategory_name_list")
)

In [ ]:
# Get distinct subcategory names
df_subcategory_name_distinc = df_subcategory_name.distinct()

In [ ]:
# Add surrogate key
df_add_sk= df_subcategory_name_distinc.withColumn("Subcategory_pk_sk", F.monotonically_increasing_id() + 1) 

In [11]:
#df_add_sk.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_subcategory")

In [ ]:
#df_subcategory_name_distinc.orderBy(F.col("subcategory_name")).show(n=200)